# Milestone 6 — Part 1: RAG Pipeline
**Model:** mistral:7b-instruct via Ollama | **Vector Store:** ChromaDB | **Embeddings:** all-MiniLM-L6-v2

In [ ]:
import os, time, json, re, statistics
from pathlib import Path
from typing import List, Dict, Tuple
import chromadb
from sentence_transformers import SentenceTransformer
import ollama
import pandas as pd
import numpy as np
print('All imports successful')
print(f'ChromaDB version: {chromadb.__version__}')

In [ ]:
DOCUMENTS = [
    {"id":"doc_01","title":"Retrieval-Augmented Generation (RAG)","content":"Retrieval-Augmented Generation (RAG) is a technique that enhances large language model outputs by grounding them in external knowledge retrieved at inference time. Unlike purely parametric models that rely solely on weights learned during training, RAG systems query a vector database to fetch relevant context before generating a response.\n\nThe RAG pipeline consists of two main phases: indexing and querying. During indexing, documents are split into chunks, converted to dense vector embeddings using an encoder model such as sentence-transformers, and stored in a vector database like FAISS or ChromaDB.\n\nDuring querying, the user question is also embedded into the same vector space. A similarity search retrieves the top-k most relevant chunks. These chunks are injected into the LLM prompt as context, enabling the model to generate factually grounded responses and reducing hallucinations.\n\nKey metrics for evaluating RAG systems include retrieval precision@k, recall@k, answer faithfulness, and answer relevance. Hallucination rate measures how often the model generates claims not supported by the retrieved context."},
    {"id":"doc_02","title":"Vector Databases and Embedding Indexes","content":"Vector databases are specialized storage systems designed to efficiently index and query high-dimensional embedding vectors. Unlike traditional relational databases, vector databases support approximate nearest neighbor (ANN) search, which enables fast semantic similarity lookup across millions of documents.\n\nFAISS (Facebook AI Similarity Search) is a widely used open-source library for ANN search. It supports several index types including Flat (exact brute-force), IVF (inverted file index for approximate search), and HNSW (hierarchical navigable small world graphs).\n\nChromaDB is a newer open-source vector database designed specifically for LLM applications. It provides a simple Python API, supports persistent storage, metadata filtering, and integrates natively with LangChain and LlamaIndex.\n\nWeaviate, Qdrant, and Milvus are production-grade vector databases with distributed architectures, offering features like multi-tenancy, hybrid search, and real-time index updates."},
    {"id":"doc_03","title":"MLOps Pipelines and CI/CD for Machine Learning","content":"MLOps applies DevOps principles to machine learning systems, enabling reliable, repeatable, and automated ML workflows in production environments.\n\nA typical MLOps pipeline includes data ingestion, data validation, feature engineering, model training, model evaluation, model registration, and model serving. Tools like Apache Airflow, Kubeflow Pipelines, and Prefect are commonly used for orchestration.\n\nContinuous Integration for ML involves automatically running data validation checks, unit tests for preprocessing code, and training pipeline smoke tests on each code commit. Continuous Deployment automates the promotion of models from staging to production after passing defined quality gates such as accuracy thresholds or latency benchmarks.\n\nModel versioning is a critical MLOps practice. Tools like MLflow, DVC, and Weights and Biases track experiment parameters, metrics, and artifact versions, enabling reproducibility and rollback to previous model versions when production issues arise."},
    {"id":"doc_04","title":"Model Monitoring and Data Drift Detection","content":"Model monitoring is the practice of continuously observing a deployed machine learning model behavior in production to detect degradation, data drift, and concept drift.\n\nData drift occurs when the statistical distribution of input features in production diverges from the distribution seen during training. Common drift detection methods include the Kolmogorov-Smirnov test, Population Stability Index (PSI), and Jensen-Shannon divergence.\n\nConcept drift refers to changes in the relationship between input features and the target variable. For example, a fraud detection model may become less effective as fraudsters adapt their strategies over time.\n\nPerformance monitoring tracks metrics like prediction latency, throughput, and model accuracy. Tools such as Evidently AI, Arize, and WhyLogs provide dashboards for real-time drift detection and alerting. When drift is detected, a retraining trigger is fired initiating an automated pipeline."},
    {"id":"doc_05","title":"LLM Serving Infrastructure and Inference Optimization","content":"Serving large language models efficiently in production requires specialized inference infrastructure that can handle high concurrency, low latency, and large model sizes.\n\nvLLM is a high-throughput LLM serving framework that uses PagedAttention to manage GPU memory efficiently. It achieves up to 24x higher throughput compared to naive Hugging Face Transformers inference by batching requests dynamically.\n\nOllama is a lightweight tool for running open-weight LLMs locally. It packages models with their runtime dependencies and exposes a simple REST API, making it ideal for development and single-machine deployments.\n\nQuantization reduces model size and inference latency by representing weights in lower precision formats such as INT8 or INT4. GGUF format used by llama.cpp enables CPU-based inference for quantized models with minimal quality degradation.\n\nKey metrics for LLM serving include time-to-first-token (TTFT), tokens per second (throughput), and GPU memory utilization."},
    {"id":"doc_06","title":"Feature Stores in Machine Learning","content":"A feature store is a centralized repository for storing, managing, and serving machine learning features. It bridges the gap between data engineering and model training and serving by providing consistent feature definitions across the ML lifecycle.\n\nFeature stores have two serving layers: the offline store used for batch training, and the online store used for real-time inference. The offline store is typically a data warehouse like BigQuery or Snowflake. The online store is typically a low-latency key-value store like Redis or DynamoDB.\n\nThe most critical problem feature stores solve is training-serving skew, the discrepancy between features computed during training and features computed during serving. By using the same feature transformation logic in both contexts, feature stores eliminate this skew.\n\nPopular feature store implementations include Feast (open-source), Tecton, Hopsworks, and Vertex AI Feature Store. They support point-in-time correct feature retrieval."},
    {"id":"doc_07","title":"Chunking Strategies for RAG Systems","content":"Chunking is the process of splitting documents into smaller segments before embedding and indexing. The chunking strategy has a significant impact on retrieval quality in RAG systems.\n\nFixed-size chunking splits documents into chunks of a predetermined token or character count such as 512 tokens with optional overlap such as 50 tokens. Overlap preserves context across chunk boundaries but increases index size.\n\nSemantic chunking uses sentence boundary detection or topic modeling to create chunks that represent coherent semantic units regardless of length. This can improve precision but requires more complex preprocessing.\n\nChunk size tradeoffs: smaller chunks (128-256 tokens) improve retrieval precision by reducing noise, but may lose surrounding context needed for generation. Larger chunks (512-1024 tokens) preserve more context but reduce retrieval precision. A common best practice is to use 512 tokens with 10-15 percent overlap."},
    {"id":"doc_08","title":"Agentic AI Systems and Tool Use","content":"Agentic AI systems are LLM-powered applications that can autonomously plan, reason, and execute multi-step tasks by calling external tools. Unlike single-turn question answering, agents maintain state across multiple reasoning steps.\n\nThe ReAct (Reasoning and Acting) framework interleaves chain-of-thought reasoning with tool invocations. At each step, the agent generates a thought, selects an action (tool call), observes the result, and iterates until the task is complete.\n\nCommon agent tools include web search, code execution, database queries, API calls, and retrieval from vector stores. Tool selection logic can be implemented as rule-based routing, LLM-based function calling, or trained classifiers.\n\nKey challenges in agentic systems include error recovery, context length management across long reasoning chains, and preventing infinite loops. Transparency and observability of agent decisions are critical for debugging and evaluation."},
    {"id":"doc_09","title":"Embedding Models for Semantic Search","content":"Embedding models convert text into dense vector representations that capture semantic meaning. Similar texts are mapped to nearby points in the embedding space, enabling semantic similarity search.\n\nThe sentence-transformers library provides a wide range of pre-trained embedding models. Popular models include all-MiniLM-L6-v2 (384 dimensions, fast and lightweight), all-mpnet-base-v2 (768 dimensions, higher quality), and multi-qa-mpnet-base-dot-v1 (optimized for question-answering retrieval).\n\nOpenAI text-embedding models are proprietary and require API access. For local and open-source deployments, BAAI/bge-large-en-v1.5 and E5-large-v2 are strong alternatives.\n\nEmbedding quality is measured on benchmarks like MTEB (Massive Text Embedding Benchmark), which evaluates models across retrieval, clustering, classification, and semantic textual similarity tasks."},
    {"id":"doc_10","title":"Evaluation Metrics for RAG and LLM Systems","content":"Evaluating RAG systems requires measuring both retrieval quality and generation quality separately, since failures can originate at either stage.\n\nRetrieval metrics include Precision@k (fraction of retrieved documents that are relevant), Recall@k (fraction of all relevant documents that are retrieved in top-k), Mean Reciprocal Rank (MRR), and Normalized Discounted Cumulative Gain (NDCG).\n\nGeneration metrics include faithfulness (whether the answer is supported by retrieved context), answer relevance (whether the answer addresses the question), and context utilization.\n\nThe RAGAS framework provides automated metrics for faithfulness, answer relevance, context precision, and context recall using an LLM as a judge.\n\nLatency is a key operational metric: retrieval latency, generation latency, and end-to-end latency should all be measured separately to identify bottlenecks."}
]
print(f'Loaded {len(DOCUMENTS)} documents')
for doc in DOCUMENTS:
    print(f"  {doc['id']}: {doc['title']} ({len(doc['content'].split())} words)")

## Chunking Strategy

**Design Decision:** Fixed-size chunking, 512-token target, 75-token overlap, paragraph-aware.

**Justification:** 512 tokens balances context preservation vs retrieval precision. 75-token overlap prevents context loss at boundaries. Paragraph-aware splitting preserves semantic units. Alternatives tested: 256-token chunks (higher precision, less context); 1024-token chunks (less precision).

In [ ]:
def chunk_document(text, chunk_size=512, overlap=75):
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks = []
    current_words = []
    for para in paragraphs:
        para_words = para.split()
        if len(current_words) + len(para_words) > chunk_size and current_words:
            chunks.append(' '.join(current_words))
            current_words = current_words[-overlap:] if len(current_words) > overlap else current_words[:]
        if len(para_words) > chunk_size:
            import re
            sentences = re.split(r'(?<=[.!?])\s+', para)
            for sentence in sentences:
                sw = sentence.split()
                if len(current_words) + len(sw) > chunk_size and current_words:
                    chunks.append(' '.join(current_words))
                    current_words = current_words[-overlap:]
                current_words.extend(sw)
        else:
            current_words.extend(para_words)
    if current_words:
        chunks.append(' '.join(current_words))
    return [c for c in chunks if len(c.strip()) > 20]

all_chunks = []
chunk_metadata = []
for doc in DOCUMENTS:
    for i, chunk in enumerate(chunk_document(doc['content'])):
        all_chunks.append(chunk)
        chunk_metadata.append({'doc_id':doc['id'],'doc_title':doc['title'],'chunk_index':i,'chunk_id':f"{doc['id']}_chunk_{i}",'word_count':len(chunk.split())})

print(f'Total chunks: {len(all_chunks)}')
print(f'Avg chunk size: {sum(m["word_count"] for m in chunk_metadata)/len(chunk_metadata):.0f} words')

## Embedding Generation

**Model:** all-MiniLM-L6-v2 (384-dim). **Justification:** Strong MTEB retrieval scores, fully open-source, fast on CPU/GPU, no API dependency.

In [ ]:
print('Loading embedding model...')
t0 = time.time()
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded in {time.time()-t0:.2f}s')
print(f'Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}')

print('Generating embeddings...')
t0 = time.time()
embeddings = embedding_model.encode(all_chunks, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
print(f'Done in {time.time()-t0:.2f}s | Shape: {embeddings.shape}')

## ChromaDB Vector Index

**Design Decision:** ChromaDB persistent client, cosine similarity. **Justification:** Persistent storage survives restarts; cosine similarity matches normalized embeddings; metadata filtering enables doc-level filtering.

In [ ]:
chroma_client = chromadb.PersistentClient(path='./chroma_db')
try:
    chroma_client.delete_collection('mlops_rag')
except:
    pass
collection = chroma_client.create_collection(name='mlops_rag', metadata={'hnsw:space':'cosine'})
t0 = time.time()
collection.add(
    ids=[m['chunk_id'] for m in chunk_metadata],
    embeddings=embeddings.tolist(),
    documents=all_chunks,
    metadatas=chunk_metadata
)
print(f'Index built in {time.time()-t0:.2f}s | Total docs: {collection.count()}')

In [ ]:
def retrieve(query, k=3):
    t0 = time.time()
    qe = embedding_model.encode([query], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=qe, n_results=k, include=['documents','metadatas','distances'])
    latency_ms = (time.time()-t0)*1000
    formatted = []
    for i in range(len(results['documents'][0])):
        formatted.append({'rank':i+1,'chunk_id':results['ids'][0][i],'doc_id':results['metadatas'][0][i]['doc_id'],'doc_title':results['metadatas'][0][i]['doc_title'],'content':results['documents'][0][i],'similarity':1-results['distances'][0][i]})
    return formatted, latency_ms

def generate_answer(query, chunks):
    context = '\n\n---\n\n'.join([f"[Source: {c['doc_title']}]\n{c['content']}" for c in chunks])
    prompt = f"""Answer the question using ONLY the provided context. If the context does not contain enough information, say so explicitly. Do not add information beyond what is in the context.\n\nCONTEXT:\n{context}\n\nQUESTION: {query}\n\nANSWER:"""
    t0 = time.time()
    response = ollama.chat(model='mistral:7b-instruct', messages=[{'role':'user','content':prompt}], options={'temperature':0.1,'num_predict':300})
    return response['message']['content'].strip(), (time.time()-t0)*1000

def run_rag(query, k=3):
    t0 = time.time()
    retrieved, ret_ms = retrieve(query, k)
    answer, gen_ms = generate_answer(query, retrieved)
    return {'query':query,'retrieved':retrieved,'answer':answer,'latency':{'retrieval_ms':round(ret_ms,1),'generation_ms':round(gen_ms,1),'end_to_end_ms':round((time.time()-t0)*1000,1)}}

# Smoke test
test = run_rag('What is RAG and how does it reduce hallucinations?')
print(f"Query: {test['query']}")
print(f"Retrieved: {[r['doc_title'] for r in test['retrieved']]}")
print(f"Answer: {test['answer'][:200]}")
print(f"Latency: {test['latency']}")

In [ ]:
EVAL_QUERIES = [
    {'id':'Q01','query':'What is retrieval-augmented generation and how does it reduce hallucinations?','relevant_doc_ids':['doc_01'],'type':'direct_factual'},
    {'id':'Q02','query':'What are the differences between FAISS and ChromaDB for vector search?','relevant_doc_ids':['doc_02'],'type':'direct_factual'},
    {'id':'Q03','query':'How does data drift detection work in ML monitoring?','relevant_doc_ids':['doc_04'],'type':'direct_factual'},
    {'id':'Q04','query':'What chunking strategies are available and what are the tradeoffs?','relevant_doc_ids':['doc_07'],'type':'direct_factual'},
    {'id':'Q05','query':'What is vLLM and how does it improve LLM serving throughput?','relevant_doc_ids':['doc_05'],'type':'direct_factual'},
    {'id':'Q06','query':'What metrics should be used to evaluate a RAG pipeline?','relevant_doc_ids':['doc_10','doc_01'],'type':'multi_doc'},
    {'id':'Q07','query':'How do feature stores prevent training-serving skew?','relevant_doc_ids':['doc_06'],'type':'direct_factual'},
    {'id':'Q08','query':'What is the best way to fine-tune a GPT model on proprietary data?','relevant_doc_ids':[],'type':'out_of_scope'},
    {'id':'Q09','query':'What embedding model should I use for semantic search?','relevant_doc_ids':['doc_09'],'type':'ambiguous'},
    {'id':'Q10','query':'How do agentic AI systems decide which tool to use for a given task?','relevant_doc_ids':['doc_08'],'type':'multi_step_reasoning'}
]
print(f'Evaluation set: {len(EVAL_QUERIES)} queries ready')

In [ ]:
def precision_at_k(retrieved, relevant, k):
    if not relevant: return 1.0 if not retrieved[:k] else 0.0
    return sum(1 for r in retrieved[:k] if r['doc_id'] in relevant) / k

def recall_at_k(retrieved, relevant, k):
    if not relevant: return 1.0
    return len(set(r['doc_id'] for r in retrieved[:k] if r['doc_id'] in relevant)) / len(relevant)

print('Running 10-query evaluation (takes ~5 minutes)...')
print('='*60)
eval_results = []
for eq in EVAL_QUERIES:
    print(f"\n{eq['id']}: {eq['query'][:55]}...")
    result = run_rag(eq['query'], k=3)
    p1 = precision_at_k(result['retrieved'], eq['relevant_doc_ids'], 1)
    p3 = precision_at_k(result['retrieved'], eq['relevant_doc_ids'], 3)
    r3 = recall_at_k(result['retrieved'], eq['relevant_doc_ids'], 3)
    eval_results.append({'query_id':eq['id'],'query':eq['query'],'type':eq['type'],'relevant_docs':eq['relevant_doc_ids'],'retrieved_doc_ids':[r['doc_id'] for r in result['retrieved']],'retrieved_titles':[r['doc_title'] for r in result['retrieved']],'answer':result['answer'],'precision_at_1':p1,'precision_at_3':p3,'recall_at_3':r3,'retrieval_ms':result['latency']['retrieval_ms'],'generation_ms':result['latency']['generation_ms'],'end_to_end_ms':result['latency']['end_to_end_ms']})
    print(f'  P@1={p1:.2f} P@3={p3:.2f} R@3={r3:.2f} | E2E={result["latency"]["end_to_end_ms"]:.0f}ms')
print('\nEvaluation complete!')

In [ ]:
df = pd.DataFrame(eval_results)
scored = df[df['type'] != 'out_of_scope']
print('='*60)
print('AGGREGATE METRICS')
print('='*60)
print(f"Mean Precision@1:        {scored['precision_at_1'].mean():.3f}")
print(f"Mean Precision@3:        {scored['precision_at_3'].mean():.3f}")
print(f"Mean Recall@3:           {scored['recall_at_3'].mean():.3f}")
print(f"Mean Retrieval Latency:  {df['retrieval_ms'].mean():.1f}ms")
print(f"Mean Generation Latency: {df['generation_ms'].mean():.1f}ms")
print(f"Mean End-to-End Latency: {df['end_to_end_ms'].mean():.1f}ms")
print(f"P95 End-to-End Latency:  {df['end_to_end_ms'].quantile(0.95):.1f}ms")
print()
print('PER-QUERY RESULTS:')
for r in eval_results:
    print(f"\n{r['query_id']} [{r['type']}]")
    print(f"  Query:     {r['query']}")
    print(f"  Expected:  {r['relevant_docs']}")
    print(f"  Retrieved: {r['retrieved_doc_ids']}")
    print(f"  P@1={r['precision_at_1']:.2f} P@3={r['precision_at_3']:.2f} R@3={r['recall_at_3']:.2f}")
    print(f"  Answer:    {r['answer'][:150]}...")

with open('eval_results.json','w') as f:
    json.dump(eval_results, f, indent=2)
print('\nSaved to eval_results.json')
print(f'\nMODEL DEPLOYMENT NOTES:')
print(f'Model: mistral:7b-instruct | Size: 7B | Serving: Ollama')
print(f'Hardware: GCP Tesla T4 15GB VRAM, CUDA 12.2')
print(f'Avg generation latency: {df["generation_ms"].mean():.0f}ms')
print(f'Avg end-to-end latency: {df["end_to_end_ms"].mean():.0f}ms')